Imports

In [5]:
import cv2
import numpy as np
import sys
import os

Variables

In [6]:
clicked_points = []
current_step = ""
MAX_DISPLAY = 900

mouse click : callback function

In [7]:
def mouse_callback(event, x, y, flags, param):
    global clicked_points
    if event== cv2.EVENT_LBUTTONDOWN:
        clicked_points.append((x,y))
        print(f"Points{len(clicked_points)} recorded: ({x},{y}")

function for resizing for display

In [8]:
def resize(img,max_dim=MAX_DISPLAY):
    h, w = img.shape[:2]
    scale = min(max_dim/w , max_dim/h , 1.0)
    if scale < 1.0:
        new_w = int(w * scale)
        new_h = int(h * scale)
        return cv2.resize(img, (new_w, new_h)),scale
    return img.copy(), 1.0

Textual instructions for drawing lines which will be displayed

In [9]:
def draw_instructions(img, lines, dot_pts=None):
    out = img.copy()
    y0 = 30
    for i, line in enumerate(lines):
        cv2.putText(out, line, (10, y0+i*28), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,0,0), 4, cv2.LINE_AA)
        cv2.putText(out, line, (10, y0+i*28), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,255,255), 2, cv2.LINE_AA)

    if dot_pts:
        labels = ["TL", "TR", "BR", "BL", "R1", "R2"]
        colors = [(0,255,0),(0,255,0),(0,255,0),(0,255,0),(255,100,0),(255,100,0)]
        for idx, (px, py) in enumerate(dot_pts):
            col = colors[idx] if idx < len(colors) else (255,255,255)
            lbl = labels[idx] if idx < len(labels) else str(idx+1)
            cv2.circle(out, (px, py), 7, col, -1)
            cv2.circle(out, (px, py), 7, (255,255,255), 2)
            cv2.putText(out, lbl, (px+10, py-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 3)
            cv2.putText(out, lbl, (px+10, py-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
    return out

collecting the number of clicks being registered

In [10]:
def collect_clicks(window_name, display_img, original_img,
                   n_clicks, instructions, scale):
    global clicked_points
    clicked_points = []

    cv2.setMouseCallback(window_name, mouse_callback)

    print(f"\n{'='*50}")
    for line in instructions:
        print(" ", line)
    print(f"  → Click {n_clicks} point(s), then press ENTER")
    print(f"{'='*50}")

    while True:
        vis = draw_instructions(display_img, instructions,
                                dot_pts=clicked_points[:n_clicks])
        # Draw connecting lines for floor corners
        if n_clicks == 4 and len(clicked_points) >= 2:
            pts_so_far = clicked_points[:min(len(clicked_points), 4)]
            for i in range(len(pts_so_far) - 1):
                cv2.line(vis, pts_so_far[i], pts_so_far[i+1], (0,255,0), 2)
            if len(pts_so_far) == 4:
                cv2.line(vis, pts_so_far[3], pts_so_far[0], (0,255,0), 2)

        # Draw line between reference points
        if n_clicks == 2 and len(clicked_points) == 2:
            cv2.line(vis, clicked_points[0], clicked_points[1], (255,100,0), 2)
            mid = ((clicked_points[0][0]+clicked_points[1][0])//2,
                   (clicked_points[0][1]+clicked_points[1][1])//2)
            cv2.putText(vis, "REF", mid, cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (255,100,0), 2)

        cv2.imshow(window_name, vis)
        key = cv2.waitKey(30) & 0xFF

        if key == 13 and len(clicked_points) >= n_clicks:  # ENTER
            break
        if key == 27:  # ESC
            print("  ✘ ESC pressed, exiting.")
            cv2.destroyAllWindows()
            sys.exit(0)

    real_pts = [(int(x / scale), int(y / scale)) for (x, y) in clicked_points[:n_clicks]]
    return real_pts

perspective transformation from user picture to top down view

In [11]:
def perspective_transform(img, floor_corners):
    src = np.float32(floor_corners)

    tl, tr, br, bl = floor_corners
    width_top    = np.linalg.norm(np.array(tr) - np.array(tl))
    width_bottom = np.linalg.norm(np.array(br) - np.array(bl))
    width        = int(max(width_top, width_bottom))

    height_left  = np.linalg.norm(np.array(bl) - np.array(tl))
    height_right = np.linalg.norm(np.array(br) - np.array(tr))
    height       = int(max(height_left, height_right))

    dst = np.float32([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1]
    ])

    M       = cv2.getPerspectiveTransform(src, dst)
    warped  = cv2.warpPerspective(img, M, (width, height))
    return warped, M, (width, height)

transformation of points

In [12]:
def transform_points(pts, M):
    pts_arr = np.float32([[p] for p in pts])
    transformed = cv2.perspectiveTransform(pts_arr, M)
    return [(int(p[0][0]), int(p[0][1])) for p in transformed]

computing the measurement with real life mesurement

In [13]:
def compute_scale(p1, p2, real_cm):
    pixel_dist = np.linalg.norm(np.array(p1) - np.array(p2))
    pixels_per_cm = pixel_dist / real_cm
    return pixels_per_cm, pixel_dist

computing the lengths of edges and pots

In [14]:
def compute_walls_and_pots(warp_size, pix_per_cm, pot_dia_cm):
    w, h = warp_size
    wall_w_cm = w / pix_per_cm
    wall_h_cm = h / pix_per_cm

    pots_along_width  = int(wall_w_cm // pot_dia_cm)
    pots_along_height = int(wall_h_cm // pot_dia_cm)
    total_pots        = pots_along_width * pots_along_height

    return {
        "wall_width_cm"     : round(wall_w_cm, 1),
        "wall_height_cm"    : round(wall_h_cm, 1),
        "pots_along_width"  : pots_along_width,
        "pots_along_height" : pots_along_height,
        "total_pots"        : total_pots,
    }

def draw_results(warped, ref_pts_warped, pix_per_cm, pot_dia_cm, stats):
    out = warped.copy()
    w, h = out.shape[1], out.shape[0]
    pot_px = int(pot_dia_cm * pix_per_cm)

    # ── draw pot grid ──
    n_cols = stats["pots_along_width"]
    n_rows = stats["pots_along_height"]
    r      = pot_px // 2

    for row in range(n_rows):
        for col in range(n_cols):
            cx = r + col * pot_px
            cy = r + row * pot_px
            cv2.circle(out, (cx, cy), r, (0, 200, 80), 2)
            cv2.circle(out, (cx, cy), 3, (0, 200, 80), -1)

    # ── draw reference line ──
    if len(ref_pts_warped) == 2:
        cv2.line(out, ref_pts_warped[0], ref_pts_warped[1], (255, 100, 0), 3)
        cv2.circle(out, ref_pts_warped[0], 6, (255,100,0), -1)
        cv2.circle(out, ref_pts_warped[1], 6, (255,100,0), -1)
        mid = ((ref_pts_warped[0][0]+ref_pts_warped[1][0])//2,
               (ref_pts_warped[0][1]+ref_pts_warped[1][1])//2)
        cv2.putText(out, "REF", (mid[0]+5, mid[1]-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,100,0), 2)

    # ── draw stats panel ──
    panel_h = 160
    panel   = np.zeros((panel_h, w, 3), dtype=np.uint8)
    panel[:] = (30, 30, 30)

    lines = [
        f"Floor:  {stats['wall_width_cm']} cm  x  {stats['wall_height_cm']} cm",
        f"Pot diameter: {pot_dia_cm} cm  |  1 pot = {pot_px} px",
        f"Pots along width:  {stats['pots_along_width']}",
        f"Pots along height: {stats['pots_along_height']}",
        f">>> TOTAL POTS:  {stats['total_pots']}",
    ]
    colors = [(220,220,220)]*4 + [(0,255,150)]
    for i, (line, col) in enumerate(zip(lines, colors)):
        cv2.putText(panel, line, (15, 25 + i*27),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2, cv2.LINE_AA)

    combined = np.vstack([out, panel])
    return combined

main function

In [15]:
def main():
    # ── load image ──
    img_path = "img_1.png"
    if not os.path.exists(img_path):
        print(f"Error: file not found → {img_path}")
        sys.exit(1)

    img = cv2.imread(img_path)
    if img is None:
        print("Error: could not read image.")
        sys.exit(1)

    print("\n🌿 Room Pot Placement Calculator")
    print("  Controls:  LEFT-CLICK to mark point  |  ENTER to confirm  |  ESC to quit\n")

    WIN = "Room Pot Calculator"
    cv2.namedWindow(WIN, cv2.WINDOW_NORMAL)

    display_img, scale = resize(img)

    instructions_corners = [
        "STEP 1/2: Click the 4 FLOOR corners",
        "Order: TOP-LEFT → TOP-RIGHT → BOTTOM-RIGHT → BOTTOM-LEFT",
        "Press ENTER when done",
    ]
    floor_pts = collect_clicks(WIN, display_img, img, 4,
                               instructions_corners, scale)

    print(f"\n  Floor corners (original px): {floor_pts}")

    instructions_ref = [
        "STEP 2/2: Click 2 endpoints of a KNOWN object",
        "Example: short edge of A4 sheet (21 cm), tile edge (60 cm)",
        "Press ENTER when done",
    ]
    ref_pts_orig = collect_clicks(WIN, display_img, img, 2,
                                  instructions_ref, scale)

    print(f"\n  Reference points (original px): {ref_pts_orig}")

    cv2.destroyAllWindows()

    print("\n─────────────────────────────────────────")
    ref_cm = float(input("  Enter real-world size of reference object (cm): "))

    while True:
        try:
            pot_dia_cm = float(input("  Enter pot diameter (cm): "))
            if pot_dia_cm > 0:
                break
        except ValueError:
            pass
        print("  ✘ Please enter a positive number.")
    print("─────────────────────────────────────────\n")

    # 4a. Perspective transform
    warped, M, warp_size = perspective_transform(img, floor_pts)

    # 4b. Transform reference points to warped space
    ref_pts_warped = transform_points(ref_pts_orig, M)

    # 4c. Scale
    pix_per_cm, pixel_ref_dist = compute_scale(
        ref_pts_warped[0], ref_pts_warped[1], ref_cm)

    print(f"   Reference: {pixel_ref_dist:.1f} px = {ref_cm} cm")
    print(f"   Scale: {pix_per_cm:.2f} pixels/cm")

    # 4d. Wall lengths & pot count
    stats = compute_walls_and_pots(warp_size, pix_per_cm, pot_dia_cm)

    print(f"\n   Floor size:  {stats['wall_width_cm']} cm × {stats['wall_height_cm']} cm")
    print(f"   Pot diameter: {pot_dia_cm} cm")
    print(f"   Pots along width:  {stats['pots_along_width']}")
    print(f"   Pots along height: {stats['pots_along_height']}")
    print(f"   TOTAL POTS: {stats['total_pots']}\n")


    result = draw_results(warped, ref_pts_warped, pix_per_cm, pot_dia_cm, stats)

    out_path = "pot_result.jpg"
    cv2.imwrite(out_path, result)
    print(f" Result saved → {out_path}")

    result_disp, _ = resize(result, max_dim=1000)
    cv2.namedWindow("Result — press any key to exit", cv2.WINDOW_NORMAL)
    cv2.imshow("Result — press any key to exit", result_disp)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()


🌿 Room Pot Placement Calculator
  Controls:  LEFT-CLICK to mark point  |  ENTER to confirm  |  ESC to quit


  STEP 1/2: Click the 4 FLOOR corners
  Order: TOP-LEFT → TOP-RIGHT → BOTTOM-RIGHT → BOTTOM-LEFT
  Press ENTER when done
  → Click 4 point(s), then press ENTER
Points1 recorded: (111,434
Points2 recorded: (470,127
Points3 recorded: (523,136
Points4 recorded: (223,461

  Floor corners (original px): [(197, 771), (835, 225), (929, 241), (396, 819)]

  STEP 2/2: Click 2 endpoints of a KNOWN object
  Example: short edge of A4 sheet (21 cm), tile edge (60 cm)
  Press ENTER when done
  → Click 2 point(s), then press ENTER
Points1 recorded: (783,289
Points2 recorded: (856,375

  Reference points (original px): [(1392, 513), (1521, 666)]

─────────────────────────────────────────
─────────────────────────────────────────

   Reference: 215.1 px = 12.0 cm
   Scale: 17.92 pixels/cm

   Floor size:  46.8 cm × 11.4 cm
   Pot diameter: 23.0 cm
   Pots along width:  2
   Pots along height: 0